This version of SLIMER implementation eliminates the line
`from src.SFT_finetuning.commons.prompter import SLIMER_instruction_prompter, Prompter`
and instead builds a function unreliant on cloning SLIMER

In [1]:
!git clone https://github.com/jyjylow/mapping_reform.git

Cloning into 'mapping_reform'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 99 (delta 31), reused 67 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 28.03 MiB | 9.02 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [13]:
os.chdir("./mapping_reform")

In [2]:
!pip3 install vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.4/383.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.0/169.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB

In [16]:
import pandas as pd

In [3]:
import os
import sys
sys.path.append(os.path.abspath("./SLIMER"))
import json


from vllm import LLM, SamplingParams
# from src.SFT_finetuning.commons.prompter import SLIMER_instruction_prompter, Prompter

# to truncate decoder prompt at 512 tokens, using SLIMER's tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("expertai/SLIMER")

INFO 07-19 10:28:58 [__init__.py:244] Automatically detected platform cuda.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [4]:
# loading SLIMER model

vllm_model = LLM("expertai/SLIMER", max_model_len=512)
# it is recommended to use a temperature of 0
# max_new_tokens can be adjusted depending on the expected length and number of entities (default 128)
sampling_params = SamplingParams(temperature=0, max_tokens=128, stop=['</s>'])

config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

INFO 07-19 10:29:32 [config.py:841] This model supports multiple tasks: {'embed', 'reward', 'generate', 'classify'}. Defaulting to 'generate'.
WARNING 07-19 10:29:32 [config.py:3320] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 07-19 10:29:32 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 07-19 10:29:32 [config.py:1472] Using max model len 512
WARNING 07-19 10:29:32 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 07-19 10:29:33 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='expertai/SLIMER', speculative_config=None, tokenizer='expertai/SLIMER', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

INFO 07-19 10:29:34 [cuda.py:311] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 07-19 10:29:34 [cuda.py:360] Using XFormers backend.
INFO 07-19 10:29:36 [parallel_state.py:1076] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
INFO 07-19 10:29:36 [model_runner.py:1171] Starting to load model expertai/SLIMER...
INFO 07-19 10:29:38 [weight_utils.py:292] Using model weights format ['*.safetensors']


model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 07-19 10:34:18 [default_loader.py:272] Loading weights took 49.10 seconds
INFO 07-19 10:34:19 [model_runner.py:1203] Model loading took 12.5524 GiB and 281.124116 seconds
INFO 07-19 10:34:22 [worker.py:294] Memory profiling takes 2.24 seconds
INFO 07-19 10:34:22 [worker.py:294] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.90) = 13.27GiB
INFO 07-19 10:34:22 [worker.py:294] model weights take 12.55GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 0.31GiB; the rest of the memory reserved for KV Cache is 0.37GiB.
INFO 07-19 10:34:22 [executor_base.py:113] # cuda blocks: 47, # CPU blocks: 512
INFO 07-19 10:34:22 [executor_base.py:118] Maximum concurrency for 512 tokens per request: 1.47x
INFO 07-19 10:34:25 [model_runner.py:1513] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in th

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 07-19 10:35:25 [model_runner.py:1671] Graph capturing finished in 60 secs, took 0.26 GiB
INFO 07-19 10:35:25 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 65.55 seconds


In [34]:

entity_definitions = {
    "event": {
        "definition": "EVENT entities refer to specific manifestations of collective action for sociopolitical movements. These can be nouns such as meetings, protests, demonstrations, lectures, and gatherings, or verbs describing these actions like met, protested, demonstrated, lectured, and gathered.",
        "guidelines": "When a verb like 'met' is present, extract the nominal form 'meeting'. Avoid labeling speculative, fictional, or allegorical events. Label only events that have occurred and are being factually recounted. "
    },
    "location": {
        "definition": "LOCATION entities refer to geographic places and specific locations, such as cities, counties, districts, and named buildings or places.",
        "guidelines": "Focus on geographic and administrative divisions, proper place names, and specific addresses. Extract as much locational information as is available. Example: extract all of 'Ship Inn, Long Lane, Bermondsey' rather than just 'Bermondsey'. Avoid labeling vague or generic spatial references."
    },
    "space": {
        "definition": "SPACE entities describe types of physical spaces, buildings, or venues where people gather or activities take place.",
        "guidelines": "Look for words like: meeting room, school, factory, hall, building, house, shop, office, church, town, square, etc. Extract these words even when they appear in proper names like 'People's School' (extract 'school') or 'Market Square' (extract 'square'). Focus on the common noun that describes the type of space."
        }
}

# The new, self-contained function
def extract_entities(input_text, max_length=512):
    """
    Extracts entities using manually formatted prompts without relying on
    the SLIMER repository's helper scripts.
    """
    print(f"INPUT TEXT: \"{input_text}\"")
    print("\nENTITIES:")

    all_outputs = {}

    for tag, dng in entity_definitions.items():

        # 1. Creating the instruction string
        instruction = (
            f"Extract the Named Entities of type {tag.upper()} from the input text. "
            f"You are given a DEFINITION and some GUIDELINES.\n"
            f"DEFINITION: {dng['definition']}\n"
            f"GUIDELINES: {dng['guidelines']}\n"
            "Return a JSON list of instances of this Named Entity type. "
            "Return an empty list if no instances are present."
        )

        # 2. Constructing the prompt scaffold
        prompt_scaffold = (
            "[INST] You are given a text chunk (delimited by triple quotes) and an instruction.\n"
            "Read the text and answer to the instruction in the end.\n"
            '"""\n'
            f"{input_text}\n"
            '"""\n'
            f"Instruction: {instruction}\n"
            "[/INST]"
        )

        # 3. Calculating token budget for input_text
        scaffold_without_inputtext = prompt_scaffold.replace("{input_text}\n", "")
        scaffold_token_count = len(tokenizer.encode(scaffold_without_inputtext))
        print(f"No. of tokens in prompt scaffold: {scaffold_token_count}")
        budget_for_input_text = max_length - scaffold_token_count - 5 # Leaving a small buffer for safety

        # 4. Tokenizing and truncating the input_text if necessary
        input_tokens = tokenizer.encode(input_text)
        if len(input_tokens) > budget_for_input_text:
            print(f"Input text is too long ({len(input_tokens)} tokens), truncating to {budget_for_input_text} tokens.")
            truncated_input_tokens = input_tokens[:budget_for_input_text]
            truncated_input_text = tokenizer.decode(truncated_input_tokens, skip_special_tokens=True)
        else:
            truncated_input_text = input_text

        # 5. Assembling the final prompt with the (potentially truncated) text
        final_prompt = prompt_scaffold.replace("{input_text_placeholder}", truncated_input_text)

        # 6. Generating responses
        responses = vllm_model.generate([final_prompt], sampling_params)
        pred_response = responses[0].outputs[0].text.strip()

        # Parse the JSON output
        try:
            entities = json.loads(pred_response) if pred_response else []
        except json.JSONDecodeError:
            entities = []  # Handle cases where the model returns non-JSON text

        all_outputs[tag] = entities

        # Format output
        if entities:
            entities_str = ", ".join([f'"{entity}"' for entity in entities])
            print(f"{tag.upper()}: [{entities_str}]")
        else:
            print(f"{tag.upper()}: []")

    return all_outputs

In [7]:
simple_test = """
Weavers and card-room hands, attend the meeting which will be held in the Charlestown meeting room, on Wednesday evening, Nov. 6th, at eight o'clock, and show by your thousands that you are determined to be no longer 'stumped upon with impunity.
"""

extract_entities(simple_test)

INPUT TEXT: "
Weavers and card-room hands, attend the meeting which will be held in the Charlestown meeting room, on Wednesday evening, Nov. 6th, at eight o'clock, and show by your thousands that you are determined to be no longer 'stumped upon with impunity.
"

ENTITIES:
No. of tokens in prompt scaffold: 302


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

EVENT: ["meeting"]
No. of tokens in prompt scaffold: 286


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

LOCATION: ["Charlestown"]
No. of tokens in prompt scaffold: 287


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

SPACE: ["Charlestown meeting room"]


{'event': ['meeting'],
 'location': ['Charlestown'],
 'space': ['Charlestown meeting room']}

In [9]:
# TO SOLVE: Input article too long! need to find a way to truncate

test_toolong = """
OLD ABERDEEN.—On Wednesday night week, a public meeting was held in the Teetotal Hall, High-street, for the purpose of forming a Chartist Association, at eight o'clock. The Hall was crowded. Mr. William Adams was called to the chair, who opened the business in an appropriate and pithy address, and introduced Mr. Nicolson, from Aberdeen. Mr. Nicolson delivered an address on the present state of the country, &c., and sat down warmly applauded. Mr. Archibald Macdonald then explained the principles of the Charter, and was followed by Mr. James Macpherson, who delivered a powerful address on the necessity of uniting in one common bond of union to overturn the unjust system of things which now exists. A gentleman named Mr. Gibb then put some questions to the speakers, which were answered to his seeming satisfaction. The National Petition, and copies of the Charter, were distributed, and an Association formed. A vote of thanks was given to the Chairman, and the meeting separated.  CHESTER.—Mr. Christopher Doyle lectured here on Thursday night week, at seven o'clock, in the Chartist Meeting Room, Steam Mill-street. Admission gratis, and free discussion was invited. The room, which will hold between 300 and 400 persons, was crowded. Thanks were voted to him at the close, and eight new members were enrolled. The National Petition was adopted at a public meeting on Monday night last."""
extract_entities(test_toolong)

INPUT TEXT: "
OLD ABERDEEN.—On Wednesday night week, a public meeting was held in the Teetotal Hall, High-street, for the purpose of forming a Chartist Association, at eight o'clock. The Hall was crowded. Mr. William Adams was called to the chair, who opened the business in an appropriate and pithy address, and introduced Mr. Nicolson, from Aberdeen. Mr. Nicolson delivered an address on the present state of the country, &c., and sat down warmly applauded. Mr. Archibald Macdonald then explained the principles of the Charter, and was followed by Mr. James Macpherson, who delivered a powerful address on the necessity of uniting in one common bond of union to overturn the unjust system of things which now exists. A gentleman named Mr. Gibb then put some questions to the speakers, which were answered to his seeming satisfaction. The National Petition, and copies of the Charter, were distributed, and an Association formed. A vote of thanks was given to the Chairman, and the meeting separated

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: The decoder prompt (length 576) is longer than the maximum model length of 512. Make sure that `max_model_len` is no smaller than the number of text tokens.

In [10]:
# this doesn't work very well #lol
# EXTRACT ENTITIES FUNCTION V2: ALL IN ONE BABY

def extract_entities_all(input_text, max_length=512):
    """
    Extracts all defined entities in a single, efficient model call
    using a comprehensive all-in-one prompt. This version has corrected indentation.
    """

    # 1. Combining all entities into one instruction
    instruction = (
        "From the text chunk, extract Named Entities for the following types:\n"
        "- EVENT: A collective action for a sociopolitical movement (e.g., meeting, protest, lecture). Extract verbs like 'met' as their noun form.\n"
        "- LOCATION: A specific geographic place (e.g., city, street, named building). Extract full proper place names.\n"
        "- SPACE: A generic type of place where people gather (e.g., room, hall, school). Extract the common noun describing the space.\n\n"
        'Return a single JSON object with keys "event", "location", and "space". Each key should have a list of strings as its value. '
        "If no instances of an entity type are found, return an empty list for that key."
    )

    # 2. Constructing prompt scaffold
    prompt_scaffold = (
        "[INST] You are given a text chunk (delimited by triple quotes) and an instruction.\n"
        "Read the text and answer to the instruction in the end.\n"
        '"""\n{input_text_placeholder}\n"""\n'
        f"Instruction: {instruction}\n[/INST]"
    )

    # 3. Calculating the token budget for the input_text
    scaffold_without_placeholder = prompt_scaffold.replace("{input_text_placeholder}\n", "")
    scaffold_token_count = len(tokenizer.encode(scaffold_without_placeholder))
    print(f"INFO: All-in-one scaffold token count: {scaffold_token_count}")
    budget_for_input_text = max_length - scaffold_token_count - 5 # Safety buffer

    # 4. Truncate the input text if it exceeds the budget
    input_tokens = tokenizer.encode(input_text)
    if len(input_tokens) > budget_for_input_text:
        print(f"INFO: Input text is too long ({len(input_tokens)} tokens), truncating to {budget_for_input_text} tokens.")
        truncated_input_tokens = input_tokens[:budget_for_input_text]
        truncated_input_text = tokenizer.decode(truncated_input_tokens, skip_special_tokens=True)
    else:
        truncated_input_text = input_text

    # 5. Assemble the final prompt
    final_prompt = prompt_scaffold.replace("{input_text_placeholder}", truncated_input_text)

    # 6. Generate a SINGLE response from the model
    responses = vllm_model.generate([final_prompt], sampling_params)
    pred_response = responses[0].outputs[0].text.strip()

    # 7. Parse the single JSON output robustly
    try:
        all_outputs = json.loads(pred_response)
        # Ensure the structure is what we expect, adding missing keys if necessary
        for key in ["event", "location", "space"]:
            if key not in all_outputs:
                all_outputs[key] = []
    except (json.JSONDecodeError, TypeError):
        print("Error: Model did not return a valid JSON object.")
        all_outputs = {"event": [], "location": [], "space": []} # Default empty structure

    print("\nENTITIES:")
    for tag, entities in all_outputs.items():
        if entities:
          entities_str = ", ".join([f'"{entity}"' for entity in entities])
          print(f"{tag.upper()}: [{entities_str}]")

        else:
            print(f"{tag.upper()}: []")

    return all_outputs

In [11]:
extract_entities_all(test_text)

INFO: All-in-one scaffold token count: 220


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Error: Model did not return a valid JSON object.

ENTITIES:
EVENT: []
LOCATION: []
SPACE: []


{'event': [], 'location': [], 'space': []}

Looks like all-in-one isn't gonna work. I'll stick to the OG for now.

In [21]:
checked_samples = pd.read_csv("./data/article_samples/q1_samples_checked.csv")

In [23]:
checked_samples.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   corpus_id                 100 non-null    int64  
 1   score                     100 non-null    float64
 2   decile                    100 non-null    int64  
 3   date                      100 non-null    object 
 4   source                    100 non-null    object 
 5   article                   100 non-null    object 
 6   is-thematically-relevant  100 non-null    bool   
 7   has-event                 100 non-null    bool   
 8   has-named-location        100 non-null    bool   
 9   has-space-type            100 non-null    bool   
 10  event-mentioned           9 non-null      object 
 11  location-mentioned        5 non-null      object 
 12  space-mentioned           4 non-null      object 
 13  notes                     12 non-null     object 
dtypes: bool(4),

In [26]:
# placeholder wrapper to catch the too-long article ValueError

def extract_entities_skiplong(article_text):
    """
    A wrapper function that calls extract_entities and returns a
    default empty dictionary if a ValueError (or any other error) occurs.
    """
    try:
        # Call your original function
        return extract_entities(article_text)
    except ValueError as e:
        # This will catch the "longer than maximum model length" error
        print(f"SKIPPING article due to ValueError: {e}")
        # Return a default structure so that json_normalize doesn't fail
        return {"event": [], "location": [], "space": []}
    except Exception as e:
        # Catch any other unexpected errors from the model
        print(f"SKIPPING article due to unexpected error: {e}")
        return {"event": [], "location": [], "space": []}

In [35]:
# # Applying extract_entities() to my 100 samples as a start..

# samples_extracted_json = checked_samples['article'].apply(extract_entities_skiplong)

# # converting json to df
# samples_extracted_df = pd.json_normalize(samples_extracted_json)

# # creating columns in the new df
# samples_extracted_df = samples_extracted_df.rename(columns={
#     'event': 'event_extracted',
#     'location': 'location_extracted',
#     'space': 'space_extracted'
# })

# samples_extracted_df = pd.concat([
#     checked_samples[['corpus_id', 'article']],
#     samples_extracted_df
# ], axis=1)

# string_columns = ['event_extracted', 'location_extracted', 'space_extracted']
# for col in string_columns:
#     # The .apply() method here works on each cell (which is a list)
#     samples_extracted_df[col] = samples_extracted_df[col].apply(
#         lambda x: ', '.join(x) if isinstance(x, list) else x
#     )

INPUT TEXT: "Mr. Donovan: If a repeal of the Corn Laws, in Yorkshire and Lancashire, were placed against the Charter, he feared it would be carried; but if they combined the Ten Hours' Bill with the Charter, they could, in all places, defeat the League. He should vote for the resolution, as it did not debar them, under circumstances, from opposing the League. He thought it best not to attend their meetings. If 100,000 Chartists were at a meeting, and only 500 Corn Law repealers, and the 500 held up their hands, and the Chartists did not, the whole meeting would be taken for Corn Law repealers. Let them pass another substantive resolution, stating that Chartists should not attend their meetings.  Mr. Webb agreed with Mr. Donovan.  Mr. McGrath was proud to see the unanimity that prevailed. With respect to the threatened scrutiny he thought the opening of the ports a matter of absolute necessity. He would sign a petition to that effect to-morrow. He would to God that the Corn Laws were er

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

SKIPPING article due to ValueError: The decoder prompt (length 561) is longer than the maximum model length of 512. Make sure that `max_model_len` is no smaller than the number of text tokens.
INPUT TEXT: "Sir J. Graham had listened with much pleasure to the speech of the hon. member for Salford, the more so as he was not till then aware that, as a factory operative, he had administered with his own hands to his own wants. ( Hear. ) That circumstance reflected great honour upon the hon. gentleman, and it afforded him (Sir J. Graham) not a little gratification to sit in that house with him on terms of perfect equality—(hear, hear)—for the speech of the hon. gentleman was of itself the most convincing proof that from the humblest classes of the community gentlemen might rise to stations of the highest importance and influence by the exercise of honest industry and unblemished integrity—(hear)—but the course of life to which the hon. gentleman owed his success had been compatible with lon

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

SKIPPING article due to ValueError: The decoder prompt (length 651) is longer than the maximum model length of 512. Make sure that `max_model_len` is no smaller than the number of text tokens.
INPUT TEXT: "HUDDERSFIELD.—Mr. Chas. Conor lectured here on Tuesday evening, and gave a cheering account of the glorious reception of the patriots in Manchester, and the defeat of the machinations of the “plague” and its minion:  GLASGOW.—A lecture was delivered in St. Ann’s Church, by Mr. Hamilton, of Stonehouse, on the evils of intemperance, and the propriety of all professing Chartists abstaining from the use of intoxicating drinks.  Gorbals.—A meeting of the inhabitants was held in their own Hall, when Mr. Currie delivered a lecture on the state of parties.  THE GLASGOW SOIREE COMMITTEE had a meeting in the L. U. S. Hall, College Open, when they entered into further arrangements for that important affair. It was also stated that the Committee had sold all the tickets which they could possibly

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

LOCATION: []
No. of tokens in prompt scaffold: 356


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

SPACE: []


In [36]:
# samples_extracted_df

,corpus_id,article,event_extracted,location_extracted,space_extracted
0,269088,"Mr. Donovan: If a repeal of the Corn Laws, in ...",,,
1,784077,Sir J. Graham had listened with much pleasure ...,,,
2,207917,HUDDERSFIELD.—Mr. Chas. Conor lectured here on...,,,
3,876329,Sometimes a commission is used apparently for ...,,,
4,153261,Listen to no rubbish. Give ear to no dissensio...,,,
...,...,...,...,...,...
95,1147763,"; fire-irons, with brass ornamented grates, fr...",,,
96,360573,"RECENT CASES. 1. William Moss, son of Thomas ...",,,
97,1168125,"THE ENGLISHMAN'S HEBREW CONCORDANCE, from £3. ...",,,
98,1289918,"man: nature, the imputed righteousness of Jesu...",,,


In [37]:
# samples_extracted_df.to_csv("./data/article_samples/extract_test.csv")